# # ERGAST
Ergast é uma API pública e gratuita que fornece dados históricos completos da Fórmula 1. É uma das fontes mais confiáveis e abrangentes para dados de F1, mantida pela comunidade desde 2008.

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import datetime
import time

In [0]:
# Databricks Notebook: /f1-project/dlt/01_bronze_ergast_dlt
# Objetivo: Pipeline DLT para ingestão de dados Ergast API nas tabelas bronze existentes

import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import datetime
import time


def fetch_ergast_data(season, endpoint):
    """Busca dados da Ergast API"""
    url = f"https://api.jolpi.ca/ergast/f1/{season}/{endpoint}.json"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        if "MRData" in data:
            if endpoint == "drivers" and "DriverTable" in data["MRData"]:
                items = data["MRData"]["DriverTable"].get("Drivers", [])
            elif endpoint == "constructors" and "ConstructorTable" in data["MRData"]:
                items = data["MRData"]["ConstructorTable"].get("Constructors", [])
            elif endpoint == "races" and "RaceTable" in data["MRData"]:
                items = data["MRData"]["RaceTable"].get("Races", [])
            elif endpoint == "results" and "RaceTable" in data["MRData"]:
                items = []
                races = data["MRData"]["RaceTable"].get("Races", [])
                for race in races:
                    race_results = race.get("Results", [])
                    for result in race_results:
                        result["raceId"] = race.get("raceId", race.get("round"))
                        result["_season"] = season
                        items.append(result)
            else:
                items = []

            # Adicionar metadados
            for item in items:
                item["_ingested_at"] = datetime.now().isoformat()
                item["_source"] = "ergast_api"
                if "_season" not in item:
                    item["_season"] = season

            return items
        return []
    except Exception as e:
        print(f"Erro ao buscar {endpoint}/{season}: {e}")
        return []

@dlt.table(
    name="bronze_drivers_raw",
    comment="Dados de pilotos da Ergast API",
    table_properties={"quality": "bronze", "pipelines.autoOptimize.managed": "true"},
)
def bronze_drivers_raw():
    """
    Pipeline DLT para ingestão de dados de pilotos
    """
    all_drivers = []
    seasons = list(range(1950, 2025))

    for season in seasons[:10]:  # Limitando para teste
        drivers = fetch_ergast_data(season, "drivers")
        all_drivers.extend(drivers)
        time.sleep(0.5)

    if all_drivers:
        df = spark.createDataFrame(all_drivers)
        return df
    else:
        # Retornar DataFrame vazio com schema correto
        return spark.createDataFrame(
            [],
            schema="""
            driverId STRING,
            driverRef STRING,
            number INT,
            code STRING,
            forename STRING,
            surname STRING,
            nationality STRING,
            dateOfBirth STRING,
            url STRING,
            _ingested_at TIMESTAMP,
            _source STRING,
            _season INT
        """,
        )

@dlt.table(
    name="bronze_constructors_raw",
    comment="Dados de construtores da Ergast API",
    table_properties={"quality": "bronze"},
)
def bronze_constructors_raw():
    """
    Pipeline DLT para ingestão de dados de construtores
    """
    all_constructors = []
    seasons = list(range(2023, 2025))

    # for season in seasons[:10]:  # Limitando para teste
    for season in seasons:
        constructors = fetch_ergast_data(season, "constructors")
        all_constructors.extend(constructors)
        time.sleep(0.5)

    if all_constructors:
        return spark.createDataFrame(all_constructors)
    else:
        return spark.createDataFrame(
            [],
            schema="""
            constructorId STRING,
            constructorRef STRING,
            name STRING,
            nationality STRING,
            url STRING,
            _ingested_at TIMESTAMP,
            _source STRING,
            _season INT
        """,
        )

@dlt.table(
    name="bronze_races_raw",
    comment="Dados de corridas da Ergast API",
    table_properties={"quality": "bronze"},
)
def bronze_races_raw():
    """
    Pipeline DLT para ingestão de dados de corridas
    """
    all_races = []
    seasons = list(range(2023, 2025))

    # for season in seasons[:10]:  # Limitando para teste
    for season in seasons:
        races = fetch_ergast_data(season, "races")
        all_races.extend(races)
        time.sleep(0.5)

    if all_races:
        return spark.createDataFrame(all_races)
    else:
        return spark.createDataFrame(
            [],
            schema="""
            raceId STRING,
            year INT,
            round INT,
            circuitId STRING,
            name STRING,
            date STRING,
            time STRING,
            url STRING,
            _ingested_at TIMESTAMP,
            _source STRING,
            _season INT
        """,
        )

@dlt.table(
    name="bronze_results_raw",
    comment="Dados de resultados da Ergast API",
    table_properties={"quality": "bronze"},
)
def bronze_results_raw():
    """
    Pipeline DLT para ingestão de dados de resultados
    """
    all_results = []
    seasons = list(range(2023, 2025))

    # for season in seasons[:5]:  # Menos temporadas pois tem mais dados
    for season in seasons:
        results = fetch_ergast_data(season, "results")
        all_results.extend(results)
        time.sleep(1)

    if all_results:
        return spark.createDataFrame(all_results)
    else:
        return spark.createDataFrame(
            [],
            schema="""
            resultId STRING,
            raceId STRING,
            driverId STRING,
            constructorId STRING,
            number INT,
            grid INT,
            position INT,
            positionText STRING,
            positionOrder INT,
            points FLOAT,
            laps INT,
            time STRING,
            milliseconds INT,
            fastestLap INT,
            rank INT,
            fastestLapTime STRING,
            fastestLapSpeed FLOAT,
            statusId STRING,
            _ingested_at TIMESTAMP,
            _source STRING,
            _season INT
        """,
        )